# KVKK — Değerlendirme (deney seti)

Bu notebook elinizdeki **hazır cevapları** değerlendirir; model çalıştırmaz, **GPU gerekmez**.

**Yüklenecek 3 dosya** (sol panel → 📁 Dosyalar → Yükle, ya da 1. hücredeki "Dosya seç" düğmesi):
1. `KVKK_6698_cevap_anahtari_120_v2.xlsx` — cevap anahtarı
2. Cevaplar dosyası — adında **cevaplar** geçmeli, `id` sütunu olmalı (ör. `KVKK_cevaplar_v2_Qwen3_8B_4bit.xlsx`)
3. `KVKK.pdf` — RAG'da kullanılan kanun metni

Hücreleri yukarıdan aşağıya sırayla çalıştırın. Klasörde adında "anahtar" ve "cevaplar" geçen **yalnızca birer** dosya olsun; eski sürümleri silin.

## 1. Dosyaları bul / yükle

In [ ]:
import os, glob
os.chdir("/content")

# Sonuçların yazılacağı klasör. Farklı bir sürümü (ör. yeni sorularla üretilmiş cevaplar)
# değerlendirirken burayı değiştirin ki önceki sonuçlar korunsun.
CIKTI = "sonuclar_40"

def bul(uzanti, kelime):
    adaylar = [p for p in glob.glob(f"**/*{uzanti}", recursive=True)
               if not p.startswith(("drive/", "sample_data/", "sonuclar"))]
    uyan = [p for p in adaylar if kelime in p.lower()]
    return uyan[0] if uyan else None

def hepsini_bul():
    return bul(".xlsx", "anahtar"), bul(".xlsx", "cevaplar"), bul(".pdf", "")

ANAHTAR, CEVAPLAR, KANUN = hepsini_bul()
if not (ANAHTAR and CEVAPLAR and KANUN):
    print("Eksik dosya var. 'Dosya seç' ile 3 dosyayı birlikte seçin (cevap anahtarı, cevaplar, KVKK PDF):")
    from google.colab import files
    files.upload()
    ANAHTAR, CEVAPLAR, KANUN = hepsini_bul()

print("Cevap anahtarı:", ANAHTAR)
print("Cevaplar      :", CEVAPLAR)
print("Kanun PDF     :", KANUN)
assert ANAHTAR and CEVAPLAR and KANUN, "Eksik dosya var; yükleyip bu hücreyi tekrar çalıştırın."

import pandas as pd
c = pd.read_excel(CEVAPLAR)
assert "id" in c.columns, "Cevaplar dosyasında 'id' sütunu yok (K01, K02... şeklinde olmalı)."
print(f"Cevaplar dosyasında {c['id'].notna().sum()} soru var ✅")

## 2. Değerlendirme kodunu hazırla

In [ ]:
!pip -q install pypdf

In [ ]:
%%writefile kvkk_degerlendirme.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
KVKK RAG / RAG'sız değerlendirme betiği
=======================================

Aşamalar (sırayla çalıştırın):

  1) retrieval : RAG'ın doğru maddeyi getirip getirmediğini kodla kontrol eder (Adım 1)
  2) judge     : Her cevabı LLM judge ile puanlar -> skor (0/0.5/1) + hata_kodu (Adım 2)
  3) ornek     : İnsan doğrulaması için dengeli, kör bir örneklem dosyası üretir
  4) uyum      : Doldurulan örneklemden insan-insan ve insan-judge uyumunu (kappa) hesaplar
  5) metrik    : Tüm metrikleri, 2x2 tabloyu, hata dağılımını ve bootstrap güven
                 aralıklarını hesaplayıp Excel + grafik olarak kaydeder (Adım 3)

Örnek kullanım:
  python kvkk_degerlendirme.py retrieval --kanun KVKK_6698.pdf
  python kvkk_degerlendirme.py judge --backend ollama --model gemma2:27b
  python kvkk_degerlendirme.py judge --backend openai --model gpt-4o --base_url https://api.openai.com/v1
  python kvkk_degerlendirme.py ornek --n 50
  python kvkk_degerlendirme.py uyum
  python kvkk_degerlendirme.py metrik

Boru hattını gerçek judge olmadan denemek için (rastgele puanlar, 'sonuclar_deneme' klasörüne):
  python kvkk_degerlendirme.py judge --backend deneme
  python kvkk_degerlendirme.py metrik --cikti sonuclar_deneme

Ortak parametreler (varsayılanları değiştirmek için):
  --anahtar  cevap anahtarı xlsx   (varsayılan: KVKK_6698_cevap_anahtari_100.xlsx)
  --cevaplar model cevapları xlsx  (varsayılan: KVKK_6698_cevaplar_100.xlsx)
  --cikti    çıktı klasörü          (varsayılan: sonuclar)

Gerekli paketler: pandas, numpy, openpyxl, requests, matplotlib  (PDF için: pypdf)
"""

import argparse
import json
import math
import os
import random
import re
import sys
import time
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Sabitler
# ---------------------------------------------------------------------------
SISTEMLER = ["baseline", "rag"]
HATA_KODLARI = ["YOK", "ESKİ", "UYDURMA", "EKSİK", "YANLIŞ_RED", "ÖNCÜL_KABUL", "ALAKASIZ"]
SKORLAR_3 = [0.0, 0.5, 1.0]
SKORLAR_5 = [0.0, 0.25, 0.5, 0.75, 1.0]
SKORLAR = SKORLAR_5          # main() içinde --olcek ile ayarlanır
OLCEK = 5
DAVRANIS_CEVAPLA = "Cevapla"
DAVRANIS_REDDET = "Reddet (belgede yok)"
DAVRANIS_ONCUL = "Öncülü düzelt"

# Cevap anahtarındaki sütun adları
S_ID, S_SORU, S_ALTIN = "ID", "Soru", "Doğru Cevap"
S_ANAHTAR, S_MADDE = "Anahtar Bilgiler (puanlama)", "Madde"
S_TIP, S_DAVRANIS = "Soru Tipi", "Beklenen Davranış"
S_PARCA, S_7499 = "Gerekli Parça Sayısı", "7499 ile Değişti mi"
S_ESKI, S_SET = "Eski Metne Göre Cevap (RAG'sız tuzak)", "Set"


# ---------------------------------------------------------------------------
# Veri yükleme
# ---------------------------------------------------------------------------
def dosya_bul(yol):
    """Dosya bulunamazsa repo düzenindeki data/ ve results/ klasörlerine de bakar."""
    if os.path.exists(yol):
        return yol
    for klasor in ["data", "results", os.path.join("..", "data"), os.path.join("..", "results")]:
        aday = os.path.join(klasor, os.path.basename(yol))
        if os.path.exists(aday):
            return aday
    sys.exit(f"Dosya bulunamadı: {yol}")


def anahtar_yukle(yol):
    df = pd.read_excel(dosya_bul(yol), sheet_name="Soru_Seti")
    df = df[df[S_ID].notna()].copy()
    df[S_ID] = df[S_ID].astype(str).str.strip()
    return df.set_index(S_ID)


def cevaplar_yukle(yol):
    df = pd.read_excel(dosya_bul(yol))
    if "id" not in df.columns:
        sys.exit("Cevaplar dosyasında 'id' sütunu yok. Baseline_vs_RAG_100.ipynb ile üretilen dosyayı kullanın "
                 "ya da soru ID'lerini (K01, K02...) 'id' sütununa ekleyin.")
    df = df[df["id"].notna()].copy()
    df["id"] = df["id"].astype(str).str.strip()
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]
    return df.set_index("id")


def bos_mu(x):
    return x is None or (isinstance(x, float) and math.isnan(x)) or str(x).strip() in ("", "—", "-", "nan")


# ---------------------------------------------------------------------------
# ADIM 1 — Retrieval kontrolü (LLM kullanılmaz)
# ---------------------------------------------------------------------------
BASLIK_RE = re.compile(r"(GEÇİCİ\s+MADDE|EK\s+MADDE|MADDE)\s+(\d+)\s*[-–]")
LISTE_RE = re.compile(r"6698\s+SAYILI\s+KANUNA\s+EK\s+VE\s+DEĞİŞİKLİK")


def madde_kodu(tur, no):
    tur = tur.upper()
    if tur.startswith("GEÇİCİ"):
        return f"G{int(no)}"
    if tur.startswith("EK"):
        return f"EK{int(no)}"
    return str(int(no))


def altin_maddeler(madde_hucresi):
    """'M9/4-c; M9/5' -> {'9'} ; 'Geçici M3/1; M18/3' -> {'G3','18'} ; 'künye' -> {'KUNYE'}"""
    if bos_mu(madde_hucresi):
        return set()
    s = str(madde_hucresi)
    if s.strip().startswith("("):        # '(M12/5 süre içermez)' gibi açıklamalar
        return set()
    out = set()
    for m in re.finditer(r"Geçici\s*M(\d+)", s, re.I):
        out.add(f"G{int(m.group(1))}")
    s2 = re.sub(r"Geçici\s*M\d+", "", s, flags=re.I)
    for m in re.finditer(r"\bM(\d+)", s2):
        out.add(str(int(m.group(1))))
    if "künye" in s.lower():
        out.add("KUNYE")
    if "değişiklik listesi" in s.lower():
        out.add("LISTE")
    return out


def normalize(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = t.replace("ﬂ", "fl").replace("ﬁ", "fi")
    return re.sub(r"\s+", " ", t).strip()


class KanunIndeksi:
    """Kanun tam metninde her karakter konumunun hangi maddeye ait olduğunu bilir."""

    def __init__(self, metin):
        self.metin = normalize(metin)
        bolgeler = [(0, "KUNYE")]
        for m in BASLIK_RE.finditer(self.metin):
            bolgeler.append((m.start(), madde_kodu(m.group(1), m.group(2))))
        lm = LISTE_RE.search(self.metin)
        if lm:
            bolgeler.append((lm.start(), "LISTE"))
        self.bolgeler = sorted(bolgeler)
        self.baslangiclar = [b[0] for b in self.bolgeler]

    def konumdaki_madde(self, pos):
        i = np.searchsorted(self.baslangiclar, pos, side="right") - 1
        return self.bolgeler[max(i, 0)][1]

    def parca_maddeleri(self, parca):
        p = normalize(parca)
        if len(p) < 20:
            return None
        bas = self.metin.find(p[:80])
        if bas < 0:
            bas = self.metin.find(p[:40])
        if bas < 0:
            return None
        son = bas + len(p)
        out = {self.konumdaki_madde(bas)}
        for b, kod in self.bolgeler:
            if bas < b < son:
                out.add(kod)
        return out


def kanun_metni_oku(yol):
    if yol.lower().endswith(".pdf"):
        from pypdf import PdfReader
        return "\n".join((p.extract_text() or "") for p in PdfReader(yol).pages)
    with open(yol, encoding="utf-8") as f:
        return f.read()


def parcalara_bol(baglam):
    if bos_mu(baglam):
        return []
    parcalar = re.split(r"\[Sayfa \d+\]", str(baglam))
    parcalar = [p.replace("=" * 20, "").strip() for p in parcalar]
    return [p for p in parcalar if p]


def parca_maddeleri_basliktan(parca):
    """Kanun dosyası verilmediğinde: yalnızca parçadaki başlıklara bakar (eksik kalabilir)."""
    out = {madde_kodu(m.group(1), m.group(2)) for m in BASLIK_RE.finditer(normalize(parca))}
    if "Kanun Numarası" in parca:
        out.add("KUNYE")
    if LISTE_RE.search(normalize(parca)):
        out.add("LISTE")
    return out if out else None


def asama_retrieval(args):
    anahtar = anahtar_yukle(args.anahtar)
    cevap = cevaplar_yukle(args.cevaplar)
    indeks = None
    if args.kanun:
        indeks = KanunIndeksi(kanun_metni_oku(args.kanun))
        print(f"Kanun metni yüklendi: {len(indeks.bolgeler)} bölge (madde) bulundu.")
    else:
        print("UYARI: --kanun verilmedi. Yalnızca parçalardaki 'MADDE n-' başlıklarına bakılacak;\n"
              "       madde ortasından başlayan parçalar tanınamaz. Doğru sonuç için kanun PDF'ini verin.")

    satirlar = []
    for qid, r in cevap.iterrows():
        if qid not in anahtar.index:
            continue
        a = anahtar.loc[qid]
        altin = altin_maddeler(a[S_MADDE])
        degerlendir = (a[S_DAVRANIS] != DAVRANIS_REDDET) and len(altin) > 0 \
            and not (not bos_mu(a[S_PARCA]) and int(a[S_PARCA]) == 0)
        getirilen, belirsiz = set(), 0
        for p in parcalara_bol(r.get("retrieved_context")):
            m = indeks.parca_maddeleri(p) if indeks else None
            if m is None:
                m = parca_maddeleri_basliktan(p)
            if m is None:
                belirsiz += 1
            else:
                getirilen |= m
        if degerlendir:
            bulunan = altin & getirilen
            recall = len(bulunan) / len(altin)
            geldi = "Evet" if recall == 1 else "Hayır"
        else:
            recall, geldi = None, None
        satirlar.append({
            "ID": qid, "altin_maddeler": ", ".join(sorted(altin)),
            "getirilen_maddeler": ", ".join(sorted(getirilen)),
            "belirsiz_parca": belirsiz, "madde_recall": recall, "rag_madde_geldi": geldi,
        })
    out = pd.DataFrame(satirlar)
    os.makedirs(args.cikti, exist_ok=True)
    yol = os.path.join(args.cikti, "retrieval.xlsx")
    out.to_excel(yol, index=False)
    d = out.dropna(subset=["rag_madde_geldi"])
    print(f"Değerlendirilen soru: {len(d)} | Hit (tüm maddeler geldi): "
          f"{(d.rag_madde_geldi == 'Evet').mean():.1%} | Ort. madde recall: {d.madde_recall.mean():.2f}")
    if out.belirsiz_parca.sum():
        print(f"Tanınamayan parça sayısı: {int(out.belirsiz_parca.sum())} (kanun dosyası verirseniz azalır)")
    print(f"Kaydedildi: {yol}")


# ---------------------------------------------------------------------------
# ADIM 2 — LLM judge
# ---------------------------------------------------------------------------
RUBRIK_3 = {
    DAVRANIS_CEVAPLA: """1   = Temel hüküm doğru, TÜM anahtar bilgiler var, sonucu değiştiren yanlış iddia yok.
0.5 = Temel hüküm doğru ve anahtar bilgilerin EN AZ YARISI var ama hepsi değil, VEYA tüm bilgiler var ama sonucu değiştirmeyen küçük bir hata var.
0   = Temel hüküm yanlış, eski kanuna göre, uydurma, ilgisiz, soru reddedilmiş ya da anahtar bilgilerin yarısından azı var.""",
    DAVRANIS_REDDET: """1   = Bilginin Kanunda bulunmadığını açıkça söylüyor ve uydurma bilgi eklemiyor.
0.5 = Kanunda olmadığını belirtiyor ama kaynağı belirsiz somut bir bilgi de ekliyor.
0   = Soruyu Kanunda varmış gibi uydurma bilgiyle cevaplıyor.""",
    DAVRANIS_ONCUL: """1   = Öncülün yanlış olduğunu açıkça söylüyor VE doğru bilgiyi veriyor.
0.5 = Doğru bilgiyi veriyor ama öncülün yanlışlığını belirtmiyor, YA DA yanlışlığı söyleyip doğrusunu vermiyor.
0   = Öncülü kabul edip ona göre cevap veriyor.""",
}

RUBRIK_5 = {
    DAVRANIS_CEVAPLA: """1    = Temel hüküm doğru, TÜM anahtar bilgiler var, yanlış bilgi yok.
0.75 = Temel hüküm doğru, anahtar bilgilerin ÇOĞU (yarısından fazlası) var, yanlış bilgi yok.
0.5  = Temel hüküm doğru ama anahtar bilgilerin önemli kısmı eksik (yarısı veya daha azı var), YA DA bilgiler var ama sonucu değiştirmeyen küçük bir hata var.
0.25 = DOĞRU YÖNDE AMA ZAYIF: konuya uygun ve altın cevapla çelişmeyen genel bir ifade var, ancak anahtar bilgi yok ya da çok az; sonucu değiştiren yanlış bilgi yok (önemsiz yan ayrıntılardaki küçük hatalar olabilir).
0    = Temel hüküm yanlış, eski (mülga) kanuna göre, ağırlıklı olarak uydurma, soruyla ilgisiz ya da cevaplanabilir soru reddedilmiş.""",
    DAVRANIS_REDDET: """1    = Bilginin Kanunda bulunmadığını açıkça söylüyor ve uydurma bilgi eklemiyor.
0.75 = Kanunda olmadığını söylüyor ve yalnızca Kanunla uyumlu genel bilgi ekliyor (uydurma somut sayı/tarih/kurum yok).
0.5  = Kanunda olmadığını belirtiyor ama kaynağı belirsiz somut bir bilgi de ekliyor.
0.25 = Belirsizlik veya emin olmadığını ifade ediyor ama yine de uydurma somut bilgi veriyor.
0    = Soruyu Kanunda varmış gibi, hiçbir çekince koymadan uydurma bilgiyle cevaplıyor.""",
    DAVRANIS_ONCUL: """1    = Öncülün yanlış olduğunu açıkça söylüyor VE doğru bilgiyi eksiksiz veriyor.
0.75 = Öncülün yanlış olduğunu açıkça söylüyor ve doğru bilginin bir kısmını veriyor.
0.5  = Doğru bilgiyi veriyor ama öncülün yanlışlığını açıkça belirtmiyor, YA DA yanlışlığı söyleyip doğrusunu vermiyor.
0.25 = Öncülü açıkça düzeltmiyor ama cevabında öncülle çelişen doğru bir ipucu var.
0    = Öncülü kabul edip ona göre cevap veriyor.""",
}
RUBRIK = RUBRIK_5

JUDGE_SISTEM = ("Sen titiz bir hukuk değerlendiricisisin. 6698 sayılı Kişisel Verilerin Korunması Kanununun "
                "7499 sayılı Kanunla değişik GÜNCEL hâline göre değerlendirme yaparsın. Kendi bilgin yerine "
                "verilen altın cevabı ve anahtar bilgileri esas alırsın. Yalnızca geçerli JSON döndürürsün.")

JUDGE_SABLON = """SORU: {soru}
BEKLENEN DAVRANIŞ: {davranis}
ALTIN CEVAP: {altin}
ANAHTAR BİLGİLER (noktalı virgülle ayrılmış): {anahtar}
ESKİ (MÜLGA) KANUNA GÖRE YANLIŞ CEVAP: {eski}

DEĞERLENDİRİLECEK CEVAP:
\"\"\"{cevap}\"\"\"

PUANLAMA RUBRİĞİ:
{rubrik}

HATA KODU (tek bir kod seç):
- YOK: cevap tam doğru (yalnızca en yüksek skorda)
- ESKİ: cevap mülga/eski kanuna göre verilmiş (yukarıdaki eski cevapla örtüşüyor)
- UYDURMA: altın cevapta olmayan yanlış bir bilgi/sayı/kurum üretilmiş
- EKSİK: doğru yönde ama anahtar bilgiler eksik
- YANLIŞ_RED: cevaplanabilir soru reddedilmiş / bilgi yok denmiş
- ÖNCÜL_KABUL: sorudaki yanlış öncül kabul edilmiş
- ALAKASIZ: soruyla ilgisiz cevap

KURALLAR:
- Anahtar bilgileri kelime eşleşmesiyle değil ANLAMLA kontrol et ("otuz gün" = "30 gün"; "aranmaz" ≠ "aranır").
- Cevabın uzunluğu ve üslubu puanı etkilemez. Kararsız kalırsan düşük puanı ver.
- Önce kısa gerekçe yaz, sonra karar ver.

Yalnızca şu JSON'u döndür:
{{"gerekce": "<en fazla 2 cümle>",
 "anahtar_bilgiler": [{{"bilgi": "<anahtar bilgi>", "var_mi": true}}],
 "skor": <{skor_secenekleri}>,
 "hata_kodu": "<YOK|ESKİ|UYDURMA|EKSİK|YANLIŞ_RED|ÖNCÜL_KABUL|ALAKASIZ>"}}"""


def judge_prompt(a, cevap_metni):
    davranis = a[S_DAVRANIS]
    return JUDGE_SABLON.format(
        soru=a[S_SORU], davranis=davranis, altin=a[S_ALTIN],
        anahtar=a[S_ANAHTAR], eski="—" if bos_mu(a.get(S_ESKI)) else a[S_ESKI],
        cevap="" if bos_mu(cevap_metni) else str(cevap_metni).strip(),
        rubrik=RUBRIK.get(davranis, RUBRIK[DAVRANIS_CEVAPLA]),
        skor_secenekleri=" | ".join(f"{x:g}" for x in SKORLAR))


def http_kontrol(r):
    if r.status_code >= 400:
        try:
            mesaj = r.json()
            mesaj = mesaj[0] if isinstance(mesaj, list) else mesaj
            mesaj = mesaj.get("error", {}).get("message", r.text)
        except Exception:
            mesaj = r.text
        raise RuntimeError(f"HTTP {r.status_code}: {str(mesaj)[:300]}")


def judge_cagir(args, sistem_msg, kullanici_msg):
    import requests
    if args.backend == "deneme":       # ağ gerektirmeyen kuru çalıştırma (boru hattını test etmek için)
        return json.dumps({"gerekce": "deneme", "anahtar_bilgiler": [],
                           "skor": random.choice(SKORLAR), "hata_kodu": random.choice(HATA_KODLARI)})
    if args.backend == "ollama":
        url = (args.base_url or "http://localhost:11434").rstrip("/") + "/api/chat"
        r = requests.post(url, timeout=args.timeout, json={
            "model": args.model, "stream": False, "format": "json",
            "options": {"temperature": 0, "seed": 42},
            "messages": [{"role": "system", "content": sistem_msg},
                         {"role": "user", "content": kullanici_msg}]})
        http_kontrol(r)
        return r.json()["message"]["content"]
    if args.backend == "openai":       # OpenAI ve OpenAI-uyumlu API'ler (Gemini, Groq, Together, vLLM...)
        url = (args.base_url or "https://api.openai.com/v1").rstrip("/") + "/chat/completions"
        anahtar = os.environ.get(args.api_key_env)
        if not anahtar:
            sys.exit(f"API anahtarı bulunamadı: {args.api_key_env} ortam değişkenini ayarlayın.")
        govde = {"model": args.model, "temperature": 0, "response_format": {"type": "json_object"},
                 "messages": [{"role": "system", "content": sistem_msg},
                              {"role": "user", "content": kullanici_msg}]}
        basliklar = {"Authorization": f"Bearer {anahtar}"}
        r = requests.post(url, timeout=args.timeout, headers=basliklar, json=govde)
        if r.status_code == 400 and "response_format" in r.text:
            govde.pop("response_format")       # JSON modunu desteklemeyen servisler için
            r = requests.post(url, timeout=args.timeout, headers=basliklar, json=govde)
        http_kontrol(r)
        return r.json()["choices"][0]["message"]["content"]
    sys.exit(f"Bilinmeyen backend: {args.backend}")


def judge_cikti_coz(metin):
    m = re.search(r"\{.*\}", metin, re.S)
    if not m:
        raise ValueError("JSON bulunamadı")
    d = json.loads(m.group(0))
    skor = float(d["skor"])
    if skor not in SKORLAR:
        raise ValueError(f"Geçersiz skor: {skor}")
    kod = str(d["hata_kodu"]).strip().upper().replace("ESKI", "ESKİ").replace("ONCUL", "ÖNCÜL") \
        .replace("YANLIS", "YANLIŞ")
    if kod not in HATA_KODLARI:
        raise ValueError(f"Geçersiz hata kodu: {kod}")
    if skor == 1.0 and kod != "YOK":
        kod = "YOK"                      # tutarlılık: tam puan alan cevapta hata olamaz
    if 0.0 < skor < 1.0 and kod == "YOK":
        kod = "EKSİK"
    if skor == 0.0 and kod == "YOK":
        raise ValueError("Tutarsız çıktı: skor 0 ama hata kodu YOK")
    return {"skor": skor, "hata_kodu": kod, "gerekce": d.get("gerekce", ""),
            "anahtar_bilgiler": d.get("anahtar_bilgiler", [])}


def asama_judge(args):
    anahtar = anahtar_yukle(args.anahtar)
    cevap = cevaplar_yukle(args.cevaplar)
    os.makedirs(args.cikti, exist_ok=True)
    onbellek_yolu = os.path.join(args.cikti, "judge_sonuclari.jsonl")

    # Önbellek: yarıda kalırsa kaldığı yerden devam eder. Bir cevap yalnızca soru metni,
    # cevap metni ve judge modeli birebir aynıysa atlanır; biri değişirse yeniden puanlanır.
    import hashlib
    def parmak_izi(qid, sistem):
        rubrik = RUBRIK.get(anahtar.loc[qid, S_DAVRANIS], "")
        metin = f"{anahtar.loc[qid, S_SORU]}\n{cevap.loc[qid, sistem]}\n{args.model}\n{OLCEK}\n{rubrik}"
        return hashlib.md5(metin.encode("utf-8")).hexdigest()

    yapilan = set()
    if os.path.exists(onbellek_yolu):
        with open(onbellek_yolu, encoding="utf-8") as f:
            for satir in f:
                if satir.strip():
                    d = json.loads(satir)
                    if d.get("parmak_izi"):
                        yapilan.add((d["ID"], d["sistem"], d["parmak_izi"]))

    isler = [(qid, s) for qid in cevap.index if qid in anahtar.index for s in SISTEMLER
             if (qid, s, parmak_izi(qid, s)) not in yapilan]
    random.Random(42).shuffle(isler)    # sıra etkisini önlemek için karışık sırada puanla
    print(f"Puanlanacak cevap: {len(isler)} (aynı soru/cevap/model ile önbellekte bulunan atlandı)")

    basarisiz, ard_arda = 0, 0
    with open(onbellek_yolu, "a", encoding="utf-8") as f:
        for i, (qid, sistem) in enumerate(isler, 1):
            prompt = judge_prompt(anahtar.loc[qid], cevap.loc[qid, sistem])
            sonuc, hata = None, None
            for deneme in range(3):
                try:
                    sonuc = judge_cikti_coz(judge_cagir(args, JUDGE_SISTEM, prompt))
                    break
                except Exception as e:
                    hata = e
                    kalici = any(k in str(e) for k in ["HTTP 400", "HTTP 401", "HTTP 403", "HTTP 404"])
                    if kalici:
                        break
                    bekle = 30 if "HTTP 429" in str(e) else 2 * (deneme + 1)   # kota: biraz bekle
                    time.sleep(bekle)
            if sonuc is None:
                basarisiz += 1
                ard_arda += 1
                print(f"  [{i}/{len(isler)}] {qid}-{sistem}: BAŞARISIZ -> {hata}")
                if ard_arda >= 3:
                    print("\nÜst üste 3 cevap puanlanamadı, durduruldu. Yukarıdaki hata mesajına bakın:\n"
                          "  HTTP 400/404 'model' -> model adı yanlış veya ücretsiz katmanda yok (--model)\n"
                          "  HTTP 401/403        -> API anahtarı hatalı veya Colab'da erişim izni kapalı\n"
                          "  HTTP 429            -> kota/dakika sınırı; birkaç dakika sonra tekrar çalıştırın")
                    break
                continue
            ard_arda = 0
            kayit = {"ID": qid, "sistem": sistem, "judge_model": args.model, "olcek": OLCEK,
                     "parmak_izi": parmak_izi(qid, sistem), **sonuc}
            f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  [{i}/{len(isler)}] {qid}-{sistem}: skor={sonuc['skor']} kod={sonuc['hata_kodu']}")
    toplam = len(judge_sonuclarini_oku(args.cikti, sessiz=True))
    print(f"\nPuanlanmış cevap: {toplam} | Bu çalıştırmada başarısız: {basarisiz}")
    if basarisiz:
        print("Eksikler için aynı hücreyi tekrar çalıştırın; puanlanmış cevaplar atlanır.")
    print(f"Kaydedildi: {onbellek_yolu}")


def judge_sonuclarini_oku(cikti, sessiz=False):
    yol = os.path.join(cikti, "judge_sonuclari.jsonl")
    kayitlar = []
    if os.path.exists(yol):
        with open(yol, encoding="utf-8") as f:
            kayitlar = [json.loads(s) for s in f if s.strip()]
    if not kayitlar:
        if sessiz:
            return pd.DataFrame(columns=["ID", "sistem"])
        sys.exit("Judge henüz hiçbir cevabı puanlamamış. Önce 'judge' adımını çalıştırın ve "
                 "çıktısında 'BAŞARISIZ' satırı olmadığından emin olun.")
    df = pd.DataFrame(kayitlar)
    # Yalnızca seçili ölçekle (--olcek) verilmiş puanları kullan; eski kayıtlarda ölçek bilgisi yoksa 3 kabul edilir
    if "olcek" not in df.columns:
        df["olcek"] = 3
    df["olcek"] = df["olcek"].fillna(3).astype(int)
    secili = df[df["olcek"] == OLCEK]
    if secili.empty and not sessiz:
        sys.exit(f"{OLCEK} basamaklı ölçekle verilmiş puan yok. Mevcut ölçekler: {sorted(df['olcek'].unique())}. "
                 f"Metrik/uyum komutuna doğru --olcek değerini verin.")
    return secili.drop_duplicates(["ID", "sistem"], keep="last")


# ---------------------------------------------------------------------------
# İnsan doğrulama örneklemi ve uyum (kappa)
# ---------------------------------------------------------------------------
def asama_ornek(args):
    anahtar = anahtar_yukle(args.anahtar)
    cevap = cevaplar_yukle(args.cevaplar)
    rng = random.Random(args.seed)
    havuz = [(qid, s) for qid in cevap.index if qid in anahtar.index for s in SISTEMLER]
    # Davranış tipi x sistem katmanlarına göre dengeli seçim (küçük gruplar ezilmesin)
    katmanlar = {}
    for qid, s in havuz:
        katmanlar.setdefault((anahtar.loc[qid, S_DAVRANIS], s), []).append((qid, s))
    for v in katmanlar.values():
        rng.shuffle(v)
    secilen, n = [], min(args.n, len(havuz))
    while len(secilen) < n:
        ilerleme = False
        for k in sorted(katmanlar):
            if katmanlar[k] and len(secilen) < n:
                secilen.append(katmanlar[k].pop())
                ilerleme = True
        if not ilerleme:
            break
    rng.shuffle(secilen)

    gizli, kor = [], []
    for i, (qid, s) in enumerate(secilen, 1):
        a = anahtar.loc[qid]
        oid = f"O{i:03d}"
        gizli.append({"ornek_id": oid, "ID": qid, "sistem": s})
        kor.append({"ornek_id": oid, "Soru": a[S_SORU], "Beklenen Davranış": a[S_DAVRANIS],
                    "Doğru Cevap": a[S_ALTIN], "Anahtar Bilgiler": a[S_ANAHTAR],
                    "Eski Kanuna Göre Cevap": "" if bos_mu(a.get(S_ESKI)) else a[S_ESKI],
                    "Değerlendirilecek Cevap": cevap.loc[qid, s],
                    "P1_skor": None, "P1_hata_kodu": None, "P2_skor": None, "P2_hata_kodu": None})
    os.makedirs(args.cikti, exist_ok=True)
    kor_yol = os.path.join(args.cikti, "insan_ornegi.xlsx")
    with pd.ExcelWriter(kor_yol) as w:
        pd.DataFrame(kor).to_excel(w, sheet_name="Puanlama", index=False)
        pd.DataFrame({"Açıklama": [
            "Her puanlayıcı yalnızca kendi sütunlarını (P1_ veya P2_) doldurur; diğerinin puanlarına bakmaz.",
            "Skor: " + ", ".join(f"{x:g}" for x in SKORLAR) + " (rubrik: 'Rubrik' sayfası).",
            "Hata kodu: " + ", ".join(HATA_KODLARI),
            "Cevabın hangi sistemden geldiği bilerek gizlenmiştir (kör değerlendirme).",
        ]}).to_excel(w, sheet_name="Talimat", index=False)
        pd.DataFrame([{"Beklenen Davranış": k, "Rubrik": v} for k, v in RUBRIK.items()]) \
            .to_excel(w, sheet_name="Rubrik", index=False)
    pd.DataFrame(gizli).to_csv(os.path.join(args.cikti, "insan_ornegi_ANAHTAR.csv"), index=False)
    print(f"{len(kor)} cevaplık kör örneklem: {kor_yol}")
    print("Puanlayıcılara 'insan_ornegi_ANAHTAR.csv' dosyasını GÖSTERMEYİN (sistem bilgisini içerir).")


def cohen_kappa(a, b, agirlikli=False):
    a, b = list(a), list(b)
    kategoriler = sorted(set(a) | set(b))
    if len(kategoriler) < 2:
        return float("nan")
    idx = {k: i for i, k in enumerate(kategoriler)}
    k = len(kategoriler)
    O = np.zeros((k, k))
    for x, y in zip(a, b):
        O[idx[x], idx[y]] += 1
    O /= O.sum()
    E = np.outer(O.sum(1), O.sum(0))
    if agirlikli:     # sıralı skorlar için karesel ağırlık (0 ile 1 arasındaki ayrışma daha ağır)
        W = np.array([[(i - j) ** 2 for j in range(k)] for i in range(k)]) / (k - 1) ** 2
    else:
        W = 1 - np.eye(k)
    pay = (W * E).sum()
    return 1 - (W * O).sum() / pay if pay > 0 else float("nan")


def asama_uyum(args):
    kor = pd.read_excel(os.path.join(args.cikti, "insan_ornegi.xlsx"), sheet_name="Puanlama")
    gizli = pd.read_csv(os.path.join(args.cikti, "insan_ornegi_ANAHTAR.csv"))
    judge = judge_sonuclarini_oku(args.cikti)
    df = kor.merge(gizli, on="ornek_id").merge(
        judge[["ID", "sistem", "skor", "hata_kodu"]], on=["ID", "sistem"], how="left")
    df = df.rename(columns={"skor": "J_skor", "hata_kodu": "J_hata_kodu"})
    for c in ["P1_skor", "P2_skor", "J_skor"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["P1_hata_kodu", "P2_hata_kodu", "J_hata_kodu"]:
        df[c] = df[c].astype(str).str.strip().str.upper().replace({"NAN": None, "NONE": None})

    df["P_skor"] = df[["P1_skor", "P2_skor"]].min(axis=1)    # anlaşmazlıkta düşük puan kuralı
    satirlar = []

    def ekle(ad, x, y):
        m = x.notna() & y.notna()
        if m.sum() == 0:
            return
        satirlar.append({"Karşılaştırma": ad, "n": int(m.sum()),
                         "Yüzde uyum": float((x[m] == y[m]).mean()),
                         "Cohen kappa": cohen_kappa(x[m], y[m]),
                         "Ağırlıklı kappa (skor)": cohen_kappa(x[m], y[m], agirlikli=True)
                         if ad.startswith("Skor") else None})

    ekle("Skor: Puanlayıcı 1 vs Puanlayıcı 2", df.P1_skor, df.P2_skor)
    ekle("Skor: Puanlayıcı 1 vs Judge", df.P1_skor, df.J_skor)
    ekle("Skor: Puanlayıcı 2 vs Judge", df.P2_skor, df.J_skor)
    ekle("Skor: İnsan (düşük olan) vs Judge", df.P_skor, df.J_skor)
    ekle("Hata kodu: Puanlayıcı 1 vs Puanlayıcı 2", df.P1_hata_kodu, df.P2_hata_kodu)
    ekle("Hata kodu: Puanlayıcı 1 vs Judge", df.P1_hata_kodu, df.J_hata_kodu)
    ekle("Hata kodu: Puanlayıcı 2 vs Judge", df.P2_hata_kodu, df.J_hata_kodu)
    sonuc = pd.DataFrame(satirlar)
    if sonuc.empty:
        sys.exit("Örneklemde puan bulunamadı. insan_ornegi.xlsx dosyasını doldurduğunuzdan emin olun.")

    yol = os.path.join(args.cikti, "uyum.xlsx")
    with pd.ExcelWriter(yol) as w:
        sonuc.to_excel(w, sheet_name="Uyum", index=False)
        anlasmazlik = df[(df.P_skor.notna()) & (df.J_skor.notna()) & (df.P_skor != df.J_skor)]
        anlasmazlik.to_excel(w, sheet_name="Judge-İnsan anlaşmazlıkları", index=False)
    pd.set_option("display.width", 160)
    print(sonuc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print("\nYorum: kappa < 0.4 zayıf, 0.4-0.6 orta, 0.6-0.8 iyi, > 0.8 çok iyi (Landis & Koch).")
    print(f"Anlaşmazlıklar incelenmek üzere kaydedildi: {yol}")


# ---------------------------------------------------------------------------
# ADIM 3 — Metrikler, bootstrap, raporlama
# ---------------------------------------------------------------------------
def gosterge_tablosu(anahtar, judge, retrieval):
    """Her soru için tek satır: iki sistemin skoru ve hata kodu yan yana."""
    p = judge.pivot(index="ID", columns="sistem", values=["skor", "hata_kodu"])
    p.columns = [f"{s}_{m}" for m, s in p.columns]
    tam = [f"{s}_{m}" for s in SISTEMLER for m in ["skor", "hata_kodu"]]
    if any(c not in p.columns for c in tam):
        sys.exit("Judge sonuçlarında iki sistemin de puanı yok. 'judge' aşamasını tamamlayın.")
    p = p.dropna(subset=tam)                                     # iki sistemi de puanlanmış sorular
    df = anahtar[[S_SET, S_TIP, S_DAVRANIS, S_7499]].join(p, how="inner")
    if retrieval is not None:
        df = df.join(retrieval.set_index("ID")[["rag_madde_geldi", "madde_recall"]], how="left")
    for s in SISTEMLER:
        df[f"{s}_skor"] = df[f"{s}_skor"].astype(float)

    # Anahtar bilgi kapsama oranı: judge'ın her anahtar bilgi için verdiği "var_mi" kararlarından
    def kapsam(liste):
        if not isinstance(liste, list) or not liste:
            return np.nan
        return sum(bool(x.get("var_mi")) for x in liste if isinstance(x, dict)) / len(liste)
    if "anahtar_bilgiler" in judge.columns:
        k = judge.assign(kapsam=judge["anahtar_bilgiler"].apply(kapsam)) \
                 .pivot(index="ID", columns="sistem", values="kapsam")
        for s in SISTEMLER:
            if s in k.columns:
                df[f"{s}_kapsam"] = k[s].reindex(df.index)
    return df


def metrik_tanimlari():
    """Her metrik: (ad, soru filtresi, soru başına değer fonksiyonu, açıklama)."""
    C = lambda d: d[S_DAVRANIS] == DAVRANIS_CEVAPLA
    R = lambda d: d[S_DAVRANIS] == DAVRANIS_REDDET
    O = lambda d: d[S_DAVRANIS] == DAVRANIS_ONCUL
    E = lambda d: d[S_7499] == "Evet"
    H = lambda d: d[S_7499] == "Hayır"
    T = lambda d: pd.Series(True, index=d.index)
    skor = lambda s: (lambda d: d[f"{s}_skor"])
    tam = lambda s: (lambda d: (d[f"{s}_skor"] == 1).astype(float))
    kismi = lambda s: (lambda d: (d[f"{s}_skor"] >= 0.5).astype(float))
    kapsam = lambda s: (lambda d: d[f"{s}_kapsam"].fillna(0.0))
    kod = lambda k: (lambda s: (lambda d: (d[f"{s}_hata_kodu"] == k).astype(float)))
    return [
        ("Genel başarı (tüm sorular, skor ort.)", T, skor, "↑"),
        ("Cevaplama — esnek doğruluk (skor ort.)", C, skor, "↑"),
        ("Cevaplama — katı doğruluk (skor=1 oranı)", C, tam, "↑"),
        ("Cevaplama — kısmen doğru ve üstü (skor ≥ 0,5 oranı)", C, kismi, "↑"),
        ("Cevaplama — anahtar bilgi kapsama oranı (ort.)", C, kapsam, "↑"),
        ("Yanlış reddetme oranı (YANLIŞ_RED)", C, kod("YANLIŞ_RED"), "↓"),
        ("Doğru reddetme (skor ort.)", R, skor, "↑"),
        ("Öncül düzeltme (skor ort.)", O, skor, "↑"),
        ("Güncel olmayan bilgi oranı (ESKİ, değişen hükümler)", E, kod("ESKİ"), "↓"),
        ("Uydurma oranı (UYDURMA, tüm sorular)", T, kod("UYDURMA"), "↓"),
        ("Değişen hükümler — skor ort.", E, skor, "↑"),
        ("Değişmeyen hükümler — skor ort.", H, skor, "↑"),
    ]


def makro_basari(d, s):
    gruplar = [d[d[S_DAVRANIS] == g][f"{s}_skor"].mean()
               for g in [DAVRANIS_CEVAPLA, DAVRANIS_REDDET, DAVRANIS_ONCUL]]
    gruplar = [g for g in gruplar if not math.isnan(g)]
    return float(np.mean(gruplar)) if gruplar else float("nan")


def paired_bootstrap(df, filtre, deger, B, rng):
    """Soruları (davranış tipine göre katmanlı) yerine koyarak yeniden örnekler;
    RAG - Baseline farkının %95 güven aralığını döndürür."""
    d = df[filtre(df)]
    if len(d) == 0:
        return (float("nan"),) * 5
    fark = (deger("rag")(d) - deger("baseline")(d)).to_numpy()
    gruplar = d[S_DAVRANIS].to_numpy()
    indeksler = [np.where(gruplar == g)[0] for g in np.unique(gruplar)]
    ortalamalar = np.empty(B)
    for b in range(B):
        secim = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in indeksler])
        ortalamalar[b] = fark[secim].mean()
    alt, ust = np.percentile(ortalamalar, [2.5, 97.5])
    # iki yönlü bootstrap p-değeri (farkın işaretinin değişme sıklığı)
    p = min(1.0, 2 * min((ortalamalar <= 0).mean(), (ortalamalar >= 0).mean()))
    return fark.mean(), alt, ust, p, len(d)


def mcnemar_kesin(df, filtre):
    """Katı doğruluk (skor=1) için kesin McNemar testi."""
    d = df[filtre(df)]
    r = d["rag_skor"] == 1
    b = d["baseline_skor"] == 1
    n01, n10 = int((~b & r).sum()), int((b & ~r).sum())
    n = n01 + n10
    if n == 0:
        return n01, n10, 1.0
    k = min(n01, n10)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
    return n01, n10, p


def asama_metrik(args):
    anahtar = anahtar_yukle(args.anahtar)
    judge = judge_sonuclarini_oku(args.cikti)
    ryol = os.path.join(args.cikti, "retrieval.xlsx")
    retrieval = pd.read_excel(ryol) if os.path.exists(ryol) else None
    if retrieval is None:
        print("Not: retrieval.xlsx yok; retrieval metrikleri atlanacak ('retrieval' aşamasını çalıştırın).")
    df = gosterge_tablosu(anahtar, judge, retrieval)
    if args.set:
        df = df[df[S_SET].astype(str).str.startswith(args.set)]
    print(f"İki sistemi de puanlanmış soru sayısı: {len(df)}")
    rng = np.random.default_rng(args.seed)

    # --- Tablo 1: ana metrikler + bootstrap ---
    satirlar = []
    for ad, filtre, deger, yon in metrik_tanimlari():
        d = df[filtre(df)]
        if len(d) == 0:
            continue
        if "kapsama" in ad and ("rag_kapsam" not in df or df["rag_kapsam"].isna().all()):
            continue
        fark, alt, ust, p, n = paired_bootstrap(df, filtre, deger, args.B, rng)
        satirlar.append({
            "Metrik": ad, "İyi yön": yon, "n (soru)": n,
            "RAG'sız": float(deger("baseline")(d).mean()), "RAG'lı": float(deger("rag")(d).mean()),
            "Fark (RAG − RAG'sız)": fark, "%95 GA alt": alt, "%95 GA üst": ust,
            "Bootstrap p": p,
            "Anlamlı mı (GA sıfırı içermiyor)": "Evet" if (alt > 0 or ust < 0) else "Hayır"})
    satirlar.append({"Metrik": "Makro başarı (3 davranış tipinin eşit ağırlıklı ort.)", "İyi yön": "↑",
                     "n (soru)": len(df), "RAG'sız": makro_basari(df, "baseline"),
                     "RAG'lı": makro_basari(df, "rag")})
    tablo1 = pd.DataFrame(satirlar)

    n01, n10, p_mc = mcnemar_kesin(df, lambda d: d[S_DAVRANIS] == DAVRANIS_CEVAPLA)
    mcnemar = pd.DataFrame([{
        "Test": "McNemar (kesin), Cevaplama — katı doğruluk",
        "Yalnız RAG doğru": n01, "Yalnız RAG'sız doğru": n10, "p": p_mc}])

    # --- Hata dağılımı ---
    hd = pd.DataFrame({s: df[f"{s}_hata_kodu"].value_counts().reindex(HATA_KODLARI, fill_value=0)
                       for s in SISTEMLER})
    hd.columns = ["RAG'sız", "RAG'lı"]
    hd_oran = hd / hd.sum()

    # --- Soru tipine göre ---
    tip = df.groupby(S_TIP).agg(n=("rag_skor", "size"), RAGsiz=("baseline_skor", "mean"),
                                RAGli=("rag_skor", "mean"))
    tip["Fark"] = tip["RAGli"] - tip["RAGsiz"]
    tip = tip.rename(columns={"RAGsiz": "RAG'sız", "RAGli": "RAG'lı"}).sort_values("Fark", ascending=False)

    # --- Retrieval: Hit ve 2x2 tablo ---
    ret_ozet, iki_iki = None, None
    if retrieval is not None and "rag_madde_geldi" in df:
        r = df.dropna(subset=["rag_madde_geldi"])
        ret_ozet = pd.DataFrame([{
            "Değerlendirilen soru": len(r),
            "Hit (gereken tüm maddeler geldi)": (r.rag_madde_geldi == "Evet").mean(),
            "Ortalama madde recall": r.madde_recall.mean()}])
        dogru = np.where(r.rag_skor == 1, "Cevap tam doğru (1)", "Cevap tam doğru değil (0 / 0.5)")
        iki_iki = pd.crosstab(r.rag_madde_geldi.map({"Evet": "Madde geldi", "Hayır": "Madde gelmedi"}),
                              pd.Series(dogru, index=r.index), margins=True, margins_name="Toplam",
                              rownames=["Retrieval"], colnames=["Cevap"])

    # --- Kaydet ---
    os.makedirs(args.cikti, exist_ok=True)
    yol = os.path.join(args.cikti, "degerlendirme.xlsx")
    with pd.ExcelWriter(yol) as w:
        tablo1.to_excel(w, sheet_name="1_Ana_metrikler", index=False)
        mcnemar.to_excel(w, sheet_name="1_Ana_metrikler", index=False, startrow=len(tablo1) + 3)
        pd.concat([hd, hd_oran.add_suffix(" (%)")], axis=1).to_excel(w, sheet_name="2_Hata_dagilimi")
        if iki_iki is not None:
            ret_ozet.to_excel(w, sheet_name="3_Retrieval", index=False)
            iki_iki.to_excel(w, sheet_name="3_Retrieval", startrow=4)
        tip.to_excel(w, sheet_name="4_Soru_tipine_gore")
        df.reset_index().rename(columns={"index": "ID"}).to_excel(w, sheet_name="5_Soru_bazinda", index=False)
        judge.to_excel(w, sheet_name="6_Judge_gerekceleri", index=False)

    grafik(hd_oran, os.path.join(args.cikti, "hata_dagilimi.png"))

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print("\n=== Ana metrikler ===")
    print(tablo1.drop(columns=["Bootstrap p"]).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"\nMcNemar (katı doğruluk): yalnız RAG doğru={n01}, yalnız RAG'sız doğru={n10}, p={p_mc:.4f}")
    print("\n=== Hata dağılımı (adet) ===")
    print(hd.to_string())
    if iki_iki is not None:
        print("\n=== Retrieval × cevap doğruluğu (RAG) ===")
        print(iki_iki.to_string())
    print(f"\nKaydedildi: {yol} ve hata_dagilimi.png")


def grafik(hd_oran, yol):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    renk = {"YOK": "#2e7d32", "EKSİK": "#9ccc65", "ESKİ": "#c62828", "UYDURMA": "#ef6c00",
            "ÖNCÜL_KABUL": "#8e24aa", "YANLIŞ_RED": "#1565c0", "ALAKASIZ": "#757575"}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    sol = np.zeros(len(hd_oran.columns))
    for kod in HATA_KODLARI:
        v = hd_oran.loc[kod].to_numpy() * 100
        ax.barh(hd_oran.columns, v, left=sol, color=renk[kod], label=kod, edgecolor="white")
        for i, x in enumerate(v):
            if x >= 6:
                ax.text(sol[i] + x / 2, i, f"%{x:.0f}", ha="center", va="center", color="white", fontsize=8)
        sol += v
    ax.set_xlim(0, 100)
    ax.set_xlabel("Cevapların oranı (%)")
    ax.set_title("Hata türlerinin dağılımı")
    ax.legend(ncol=4, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.28), frameon=False)
    for k in ["top", "right"]:
        ax.spines[k].set_visible(False)
    fig.tight_layout()
    fig.savefig(yol, dpi=200)
    plt.close(fig)


# ---------------------------------------------------------------------------
# Komut satırı
# ---------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description="KVKK RAG / RAG'sız değerlendirme",
                                 formatter_class=argparse.RawDescriptionHelpFormatter, epilog=__doc__)
    ap.add_argument("asama", choices=["retrieval", "judge", "ornek", "uyum", "metrik"])
    ap.add_argument("--anahtar", default="KVKK_6698_cevap_anahtari_100.xlsx")
    ap.add_argument("--cevaplar", default="KVKK_6698_cevaplar_100.xlsx")
    ap.add_argument("--cikti", default="sonuclar")
    ap.add_argument("--kanun", help="retrieval: kanun metni (PDF veya TXT) — madde eşlemesi için önerilir")
    ap.add_argument("--backend", default="ollama", choices=["ollama", "openai", "deneme"],
                    help="judge: ollama (yerel), openai (OpenAI-uyumlu API) veya deneme (kuru çalıştırma)")
    ap.add_argument("--model", help="judge: hakem model adı (değerlendirilen modelden farklı ailede olmalı)")
    ap.add_argument("--base_url", help="judge: API adresi (ör. http://localhost:11434)")
    ap.add_argument("--api_key_env", default="OPENAI_API_KEY", help="judge: API anahtarının ortam değişkeni")
    ap.add_argument("--timeout", type=int, default=180)
    ap.add_argument("--n", type=int, default=50, help="ornek: insan örneklemi büyüklüğü")
    ap.add_argument("--B", type=int, default=10000, help="metrik: bootstrap tekrar sayısı")
    ap.add_argument("--set", help="metrik: yalnızca bu seti raporla (ör. 'Final' veya 'Deney')")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--olcek", type=int, choices=[3, 5], default=5,
                    help="judge/ornek: puan ölçeği. 5 = 0/0.25/0.5/0.75/1 (varsayılan), 3 = 0/0.5/1")
    args = ap.parse_args()

    global SKORLAR, RUBRIK, OLCEK
    OLCEK = args.olcek
    SKORLAR, RUBRIK = (SKORLAR_5, RUBRIK_5) if OLCEK == 5 else (SKORLAR_3, RUBRIK_3)

    if args.asama == "judge" and args.backend == "deneme" and args.cikti == "sonuclar":
        args.cikti = "sonuclar_deneme"   # rastgele deneme puanları gerçek sonuçlara karışmasın
        print("Deneme modu: çıktılar 'sonuclar_deneme' klasörüne yazılacak "
              "(diğer aşamalarda --cikti sonuclar_deneme kullanın).")
    if args.asama == "judge" and args.backend != "deneme" and not args.model:
        ap.error("judge aşaması için --model gerekli")
    if args.asama == "judge" and args.model and "qwen" in args.model.lower():
        print("UYARI: Değerlendirilen model qwen ailesinden; hakem olarak farklı bir aile önerilir.")

    {"retrieval": asama_retrieval, "judge": asama_judge, "ornek": asama_ornek,
     "uyum": asama_uyum, "metrik": asama_metrik}[args.asama](args)


if __name__ == "__main__":
    main()

## 3. Adım 1 — Retrieval: RAG doğru maddeyi getirdi mi?
LLM kullanılmaz; kod, getirilen parçaların kanunun hangi maddesine ait olduğunu PDF'ten bulur.

In [ ]:
!python kvkk_degerlendirme.py retrieval --anahtar "$ANAHTAR" --cevaplar "$CEVAPLAR" --kanun "$KANUN" --cikti "$CIKTI"

import pandas as pd
pd.read_excel(f"{CIKTI}/retrieval.xlsx")

## 4. Adım 2 — LLM judge ile puanlama
1. [Google AI Studio](https://aistudio.google.com/)'dan ücretsiz bir API anahtarı alın.
2. Sol paneldeki **🔑 Secrets** bölümüne `GEMINI_API_KEY` adıyla ekleyin ve "Not defteri erişimi"ni açın.
3. 4a hücresi anahtarı test eder ve kullanabileceğiniz modelleri listeler; 4b hücresi judge modelini kendisi seçer.

Yarıda kesilirse hücreyi tekrar çalıştırın; puanlanmış cevaplar atlanır.

### 4a. Bağlantı testi: anahtar çalışıyor mu, hangi modeller kullanılabilir?

In [ ]:
import requests
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
KEY = os.environ["GEMINI_API_KEY"]

r = requests.get("https://generativelanguage.googleapis.com/v1beta/models", params={"key": KEY})
if r.status_code != 200:
    print("❌ API anahtarı çalışmıyor:", r.status_code, r.text[:300])
else:
    modeller = [m["name"].replace("models/", "") for m in r.json().get("models", [])
                if "generateContent" in m.get("supportedGenerationMethods", [])]
    print("✅ Anahtar çalışıyor. Kullanılabilir Flash modelleri:")
    for m in modeller:
        if "flash" in m:
            print("  ", m)

### 4b. Judge'ı çalıştır
Aşağıdaki modeller sırayla denenir; ilk yanıt veren judge olarak kullanılır. Bütün cevaplar **aynı** modelle puanlanmalıdır.
Yarıda kesilirse hücreyi tekrar çalıştırın: aynı soru, cevap ve modelle puanlanmış cevaplar atlanır, değişenler yeniden puanlanır.

In [ ]:
import time
ADAYLAR = ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite"]   # sürüm numaralı modeller (…-latest kullanmayın)

JUDGE_MODEL = None
for model in ADAYLAR:
    for deneme in range(3):
        r = requests.post("https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
                          headers={"Authorization": f"Bearer {KEY}"},
                          json={"model": model, "messages": [{"role": "user", "content": 'Sadece {"ok": true} yaz.'}]})
        print(f"{model} deneme {deneme + 1}:", r.status_code)
        if r.status_code == 200:
            JUDGE_MODEL = model
            break
        time.sleep(20)
    if JUDGE_MODEL:
        break
assert JUDGE_MODEL, "Modeller şu an yanıt vermiyor (yoğunluk/kota); 10-15 dakika sonra tekrar deneyin."
print("✅ Judge modeli:", JUDGE_MODEL, "| Sonuç klasörü:", CIKTI)

# Kota dolmasın diye her puanlamadan sonra 13 sn bekleme (bir kez eklenir)
kod = open("kvkk_degerlendirme.py", encoding="utf-8").read()
if "time.sleep(13)" not in kod:
    kod = kod.replace("            ard_arda = 0\n", "            ard_arda = 0\n            time.sleep(13)\n", 1)
    open("kvkk_degerlendirme.py", "w", encoding="utf-8").write(kod)

!python kvkk_degerlendirme.py judge --anahtar "$ANAHTAR" --cevaplar "$CEVAPLAR" \
    --backend openai --base_url https://generativelanguage.googleapis.com/v1beta/openai/ \
    --api_key_env GEMINI_API_KEY --model "$JUDGE_MODEL" --cikti "$CIKTI"

## 5. Adım 3 — Metrikler, bootstrap güven aralıkları, hata dağılımı
Puanlama 5 basamaklıdır (0 / 0,25 / 0,5 / 0,75 / 1). Raporlanan ek ölçüler: **kısmen doğru ve üstü** (skor ≥ 0,5 oranı) ve **anahtar bilgi kapsama oranı**. Eski 3 basamaklı puanlarla metrik almak için komuta `--olcek 3` ekleyin.

In [ ]:
!python kvkk_degerlendirme.py metrik --anahtar "$ANAHTAR" --cevaplar "$CEVAPLAR" --cikti "$CIKTI"

from IPython.display import Image, display
if os.path.exists(f"{CIKTI}/hata_dagilimi.png"):
    display(Image(f"{CIKTI}/hata_dagilimi.png"))

## 6. İnsan doğrulaması (judge'ı kontrol etmek için)
Aşağıdaki hücre cevaplardan 30'unu dengeli ve **kör** (sistem adı gizli) seçer. Puanlama ölçeği judge ile aynıdır (0 / 0,25 / 0,5 / 0,75 / 1); rubrik indirilen dosyanın 'Rubrik' sayfasındadır.
1. Hücreyi çalıştırın, inen `insan_ornegi.xlsx` dosyasını iki kişi ayrı ayrı doldursun (P1_ ve P2_ sütunları).
2. Doldurulan dosyayı sol panelden sonuç klasörüne (`CIKTI`, varsayılan `sonuclar_v2/`) geri yükleyin (aynı adla, üzerine yazarak).
3. Sonraki hücreyle uyumu (kappa) hesaplayın.

In [ ]:
!python kvkk_degerlendirme.py ornek --anahtar "$ANAHTAR" --cevaplar "$CEVAPLAR" --n 30 --cikti "$CIKTI"

from google.colab import files
files.download(f"{CIKTI}/insan_ornegi.xlsx")

In [ ]:
!python kvkk_degerlendirme.py uyum --cikti "$CIKTI"

## 7. Tüm sonuçları indir

In [ ]:
!zip -q -r "$CIKTI.zip" "$CIKTI"
from google.colab import files
files.download(f"{CIKTI}.zip")